# Kaggle End-to-End (2xT4 with fallback)

This notebook downloads the dataset from the HF Hub, prepares a 50k sample with age prefixes, trains GPT-2, evaluates readability, saves the model, and uploads to your Hugging Face repo.

If 2 GPUs are available, it uses DataParallel. Otherwise it falls back to a single GPU or CPU.

In [ ]:
import os
import sys
import subprocess
import torch

print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))

BASE_DIR = '/kaggle/working'
RAW_DIR = os.path.join(BASE_DIR, 'data', 'raw_hf')
PROC_DIR = os.path.join(BASE_DIR, 'data', 'processed')
MODEL_OUT = os.path.join(BASE_DIR, 'model_output')
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(MODEL_OUT, exist_ok=True)

In [ ]:
REPO_URL = 'https://github.com/khedimyoucef/NLP-MINI-PROJECT.git'
REPO_DIR = 'NLP-MINI-PROJECT'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL], check=True)
sys.path.insert(0, REPO_DIR)
print('Repo ready:', REPO_DIR)

In [ ]:
!pip -q install -r NLP-MINI-PROJECT/requirements.txt
!pip -q install huggingface_hub

In [ ]:
import os
from huggingface_hub import login

token = os.environ.get('HF_TOKEN')
if token:
    login(token=token)
    print('Logged in with HF_TOKEN')
else:
    print('HF_TOKEN not found; you can still run, but upload will ask for login')

In [ ]:
from datasets import load_dataset

ds = load_dataset("ajibawa-2023/Children-Stories-Collection", split='train', cache_dir=os.path.join(BASE_DIR, 'hf_cache'))
print('Columns:', ds.column_names)
print('Rows:', len(ds))
ds.save_to_disk(RAW_DIR)
print('Saved raw dataset to', RAW_DIR)
print(ds[0])

In [ ]:
from datasets import Dataset

PREFIXES = [
    'Level: Age3-4 — Simple words.',
    'Level: Age5-6 — Short sentences.',
    'Level: Age7-8 — Moderate vocabulary.',
    'Level: Age9-10 — Longer sentences.',
    'Level: Age11-12 — Richer vocabulary.'
]

def to_text(example):
    if 'text' in example and example['text']:
        return example['text']
    if 'story' in example and example['story']:
        return example['story']
    prompt = example.get('prompt', '')
    body = example.get('completion', '') or example.get('response', '') or example.get('text', '')
    return (prompt + ' ' + body).strip()

sample_size = 50000
ds_sample = ds.shuffle(seed=42).select(range(min(sample_size, len(ds))))

def add_prefix(example, idx):
    prefix = PREFIXES[idx % len(PREFIXES)]
    text = to_text(example)
    return {'text': f'{prefix} {text}'}

ds_pref = ds_sample.map(add_prefix, with_indices=True, remove_columns=ds_sample.column_names)
out_path = os.path.join(PROC_DIR, 'prefix_stories.json')
ds_pref.to_json(out_path)
print('Saved', out_path)
print(ds_pref[0]['text'][:200])

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

ds_tok = load_dataset('json', data_files=out_path, split='train')

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=256)

ds_tok = ds_tok.map(tokenize, batched=True, remove_columns=['text'])
ds_tok = ds_tok.train_test_split(test_size=0.05, seed=42)
train_ds = ds_tok['train']
val_ds = ds_tok['test']
train_ds.set_format('torch')
val_ds.set_format('torch')
print('Train size:', len(train_ds), 'Val size:', len(val_ds))

In [ ]:
import math
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device_count = torch.cuda.device_count()

model = AutoModelForCausalLM.from_pretrained('gpt2')
model = model.to(device)
if device_count > 1:
    model = torch.nn.DataParallel(model)

per_device_batch = 4
grad_accum = 4
epochs = 2
lr = 2e-5

train_loader = DataLoader(train_ds, batch_size=per_device_batch, shuffle=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler() if use_amp else None

print('Training...')
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch['input_ids']
        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs = model(**batch, labels=labels)
            loss = outputs.loss
            if loss.ndim > 0:
                loss = loss.mean()
            loss = loss / grad_accum
        if scaler:
            scaler.scale(loss).backward()
        else:
            loss.backward()
        if step % grad_accum == 0:
            if scaler:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        total_loss += float(loss.detach().cpu())
        if step % 50 == 0:
            print(f'Epoch {epoch} Step {step} Loss {total_loss/step:.4f}')
    print(f'Epoch {epoch} done, avg loss {total_loss/step:.4f}')

In [ ]:
# Evaluation: perplexity
from torch.utils.data import DataLoader
model.eval()
val_loader = DataLoader(val_ds, batch_size=8)
losses = []
with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch['input_ids']
        outputs = model(**batch, labels=labels)
        loss = outputs.loss
        if loss.ndim > 0:
            loss = loss.mean()
        losses.append(float(loss.detach().cpu()))
avg_loss = sum(losses) / len(losses)
ppl = math.exp(avg_loss)
print('Validation loss:', avg_loss)
print('Perplexity:', ppl)

In [ ]:
# Readability check across age prefixes
import textstat
gen_model = model.module if hasattr(model, 'module') else model
theme = 'a dragon afraid of fire'
for prefix in PREFIXES:
    prompt = prefix + ' ' + theme
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    out = gen_model.generate(**inputs, max_new_tokens=120, do_sample=True, temperature=0.8)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    print(prefix, 'FK grade:', textstat.flesch_kincaid_grade(text))

In [ ]:
# Save model and tokenizer
gen_model = model.module if hasattr(model, 'module') else model
gen_model.save_pretrained(MODEL_OUT)
tokenizer.save_pretrained(MODEL_OUT)
print('Saved model to', MODEL_OUT)

In [ ]:
# Upload to Hugging Face Hub
from huggingface_hub import upload_folder
upload_folder(folder_path=MODEL_OUT, repo_id='khedim/NLP-MINI-PROJECT', repo_type='model')
print('Uploaded to Hugging Face: khedim/NLP-MINI-PROJECT')